# **League of Legends. Этап 1 - Extract. Извлечение данных с использованием официального API Riot Games**

15.06.2026

## Цель:

1. Реализовать	сборщик	данных,	который собирает	игроков	из	лиг	Challenger,	Grandmaster,	Master	для	
регионов	euw1	и	na1.	
2.  Для	каждого	игрока	загружает	до	100	идентификаторов	его	рейтинговых	матчей	за	текущий	месяц.	
3. По	каждому	матчу	загружает	детальную	информацию	(участники,	убийства,	смерти,	вспоможения,	победа/поражение).	

Учитывать	rate	limits	API:	20	запросов	в	секунду,	100	запросов	за	120	секунд.	

Реализовать	планировщик	с	очередью	времени.

## Импорт/установка библиотек

In [2]:
# Устанавливаем  библиотеки: для отправки HTTP-запросов (нужна для API и парсинга сайтов)
#!chcp 65001
#pip install -q requests 
#pip install python-dotenv

In [3]:
import os
from dotenv import load_dotenv
from pathlib import Path
import logging # для логирования

In [4]:
import gc # Garbage Collector Сборщик мусора для очистки памяти
import json
import requests   # для HTTP-запросов к API
import pandas as pd  # для работы с таблицами
import time          # чтобы делать паузы между запросами и не получить бан от API
from datetime import datetime, timedelta, timezone

## Предварительные настройки

In [6]:
# базовый путь
base_dir = Path(os.getcwd()) 

In [7]:
# Загрузка личного токена из файла настройки окружения lol.env
env_file = base_dir / "lol.env"
if env_file.is_file():
    # Загружаем токен, передавая полный путь
    load_dotenv(env_file)
    display("Файл успешно найден и загружен!")
    RIOT_API_KEY = os.getenv("RIOT_API_KEY")
else:
    display(f"❌ Ошибка: Файла нет в папке {base_dir}.")

# Выводим результат
display(f"Токен Riot API состоит из : {len(RIOT_API_KEY)} символов") 

'Файл успешно найден и загружен!'

'Токен Riot API состоит из : 42 символов'

In [8]:
# закголовок для запроса
HEADERS = {"X-Riot-Token": RIOT_API_KEY}

In [7]:
# Список регионов для сбора: euw1 - сервер Europe West (Западная Европа), na1 - North America (Северная Америка)
array_regions = ["euw1", "na1"]  

In [8]:
# Список лиг 
array_leagues = ["challenger", "grandmaster", 'master']  

In [9]:
# Очередь - из какой турнирной таблицы (очереди) нужно забрать список игроков. 
# RANKED_SOLO_5x5 - самый популярный и престижный соревновательный режим, где игроки заходят в матч в одиночку или вдвоем с другом.
QUEUE = "RANKED_SOLO_5x5"

In [10]:
def get_route(region):
    """ Функция для определения глобального маршрута (ROUTE) по региону, т.к. в некоторых эндпоинтах требуется полное название сервера 
    """
    if region =="na1":
        return "americas" 
    else: return "europe"

## Сбор игроков из лиг Challenger, Grandmaster, Master для регионов euw1 и na1

Рассматривемые лиги:
- **Challenger — Лига Претендентов**
- **Grandmaster — Лига Грандмастеров**
- **Master — Лига Мастеров**
  
**GET /lol/league/v4/challengerleagues/by-queue/{queue}** возвращает игроков только из лиги Challenger. Ввозвращает объект типа LeagueListDTO - метаданные лиги конкретного региона и массив всех топ-игроков - список лучших игроков сервера (до 300 человек в зависимости от региона).

**GET /lol/league/v4/grandmasterleagues/by-queue/{queue}** — возвращает игроков для лиги Grandmaster.

**GET /lol/league/v4/masterleagues/by-queue/{queue}** — возвращает игроков для лиги Master.

Структура ответов для всех трёх лиг абсолютно одинаковая. Сервер Riot Games возвращает идентичный JSON-объект для каждого из этих эндпоинтов. Различаются только текстовые значения в полях **tier** и **name**.

**Расшифровка полей верхнего уровня**
- tier (string): Ранг лиги. 
- leagueId (string): Уникальный UUID-идентификатор конкретной лиги в системе Riot Games.
- queue (string): Игровой режим. Самые частые значения: RANKED_SOLO_5x5 (одиночная/парная очередь) или RANKED_FLEX_SR (гибкая очередь).
- name (string): Внутреннее лорное название лиги, генерируемое игрой.  
- entries (list): Массив объектов с игроками, отсортированный хаотично.

**Структура объектов внутри entries**
- puuid (string): ID игрока,
- leaguePoints (int): Текущее количество очков в лиге (LP). 
- rank (string): Внутритировый дивизион. Для Challenger всегда равен "I".
- wins (int): Количество побед игрока в текущем сезоне.
- losses (int): Количество поражений в текущем сезоне.
- hotStreak (boolean): Флаг серии побед. true, если игрок победил в 3 или более играх подряд.
- veteran (boolean): Находится ли игрок в этой лиге более 100 игр.
- freshBlood (boolean): Новичок лиги. Показывает, попал ли игрок в Challenger совсем недавно.
- inactive (boolean): Флаг неактивности (аккаунт теряет очки за пропуск игр). 

In [11]:
# Список, куда мы будем складывать датафреймы по каждой лиге
all_dfs = []

In [12]:
# Полностью очищаем старые обработчики, чтобы избежать конфликтов в Jupyter
#for handler in logging.root.handlers[:]:
#    logging.root.removeHandler(handler)

In [13]:
log_file = base_dir /  'all_players_data.log'
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,  #Сбрасывает настройки Jupyter
)

In [14]:
def gathering_of_league_players(region, league_name, queue, my_headers, all_dfs_list):
    """
    Используя API /lol/league/v4/{league_name}leagues/by-queue/{queue}
    собираем puuid игроков региона region и лиги league_name.
    Все локальные переменные стираются из памяти на выходе.
    """
    print(f"Запрос списка игроков из лиги: {league_name.upper()}...")
    logging.info(f"Запрос списка игроков из лиги: {league_name.upper()}...")
    
    # Формируем URL в зависимости от лиги
    # Эндпоинты: /challengerleagues/, /grandmasterleagues/, /masterleagues/
        
    #url = f"https://{region}://{league}leagues/by-queue/{QUEUE}"
    url = f"https://{region}.api.riotgames.com/lol/league/v4/{league_name}leagues/by-queue/{queue}"
    print("Итоговый URL:", url)

    time.sleep(1.6)  # Пауза перед каждым запросом - максимум 100 запросов за 120 секунд.
        
    try:
        response = requests.get(url, headers=my_headers, timeout=5) # запрос
        
        # Если ключ устарел (403) или лимит исчерпан (429), этот блок защитит от вылета
        if not response.ok:
            print(f"❌ Сбой запроса {region.upper()} {league_name}: Код {response.status_code}")
            logging.error(f"❌ Сбой запроса {region.upper()} {league_name}: Код {response.status_code}")
            print(f"Ответ сервера: {response.text}")
            logging.error(f"Ответ сервера: {response.text}")
            return

        data = response.json()
        
         # Полезная информация о лиге
        print(f"\n Успешно получено!")
        logging.info("Успешно получено!")
        print(f"Лига: {data.get("name")}")
        logging.info(f"Лига: {data.get("name")}")
        print("Тир:",    data.get("tier"))
        logging.info(f"Тир: {data.get("tier")}")
        
        # 'entries' — список игроков в этой лиге
        # Каждый элемент — словарь с данными об одном игроке
        entries = data.get("entries", [])
        players = data["entries"]

        print("Количество игроков:", len(players))
        logging.info(f"Количество игроков: {len(players)}")
        
        league_matches = set() # Сюда собираем уникальные ID матчей за май

        # Создаём временный DataFrame для текущей лиги и региона
        current_df = pd.DataFrame(entries)

        # добавляем столбцы регион и лига
        current_df['league_type'] = league_name
        current_df['region'] = region

        # Добавляем текущий датафрейм в общий список
        all_dfs.append(current_df)
        print(f"Добавлено игроков: {len(current_df)}")
        logging.info(f"Добавлено игроков: {len(current_df)}")
            
    except requests.exceptions.Timeout:
        print(f"❌ Превышено время ожидания сервера для {region}!")
        logging.error(f"❌ Превышено время ожидания сервера для {region}!")
        return    
    except requests.exceptions.RequestException as e:
        print(f"Ошибка соединения с {region}: {e}")
        logging.error(f"Ошибка соединения с {region}: {e}")
        return
        


In [15]:
# Идем по регионам
logging.info("Скрипт запущен")

for region in array_regions:
    route = get_route(region)
    print(f"\n=== Начало сбора данных для региона: {region.upper()} ===")
    logging.info(f"\n=== Начало сбора данных для региона: {region.upper()} ===")
    
    # Идем по лигам
    for league in array_leagues:
        gathering_of_league_players(region, league, QUEUE, HEADERS, all_dfs)
        
        gc.collect() 
        
        # Задержка для соблюдения Rate Limit
        time.sleep(2)
        
# Проверяем, удалось ли собрать хоть какие-то данные
if all_dfs:
    # Объединяем все маленькие таблицы в один огромный DataFrame
    final_df = pd.concat(all_dfs, ignore_index=True)

    # Сортируем игроков по очкам внутри их региона и лиги
    if "leaguePoints" in final_df.columns:
        final_df = final_df.sort_values(by=["region", "league_type", "leaguePoints"], 
                                         ascending=[True, True, False]).reset_index(drop=True)

    print("\n=========================================")
    print(f"Сбор завершен! Общий размер таблицы: {final_df.shape}")
    logging.info(f"Сбор завершен! Общий размер таблицы: {final_df.shape}")
    print("=========================================")
    
    # Сохраняем один общий файл на диск
    output_file = os.path.join(base_dir, "all_players_data.csv") if base_dir else "all_players_data.csv"
    final_df.to_csv(output_file, index=False)
    print(f"Все данные успешно сохранены в файл: {output_file}")
    logging.info(f"Все данные успешно сохранены в файл: {output_file}")
else:
    print("\n❌ Не удалось собрать данные. Общая таблица пуста.")       
    logging.error("\n❌ Не удалось собрать данные. Общая таблица пуста.")


=== Начало сбора данных для региона: EUW1 ===
Запрос списка игроков из лиги: CHALLENGER...
Итоговый URL: https://euw1.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5

 Успешно получено!
Лига: None
Тир: CHALLENGER
Количество игроков: 317
Добавлено игроков: 317
Запрос списка игроков из лиги: GRANDMASTER...
Итоговый URL: https://euw1.api.riotgames.com/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5

 Успешно получено!
Лига: None
Тир: GRANDMASTER
Количество игроков: 800
Добавлено игроков: 800
Запрос списка игроков из лиги: MASTER...
Итоговый URL: https://euw1.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5

 Успешно получено!
Лига: None
Тир: MASTER
Количество игроков: 10000
Добавлено игроков: 10000

=== Начало сбора данных для региона: NA1 ===
Запрос списка игроков из лиги: CHALLENGER...
Итоговый URL: https://na1.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5

 Успешно получено!
Лига: None
Тир: CHALLEN

In [16]:
logger = logging.getLogger()

# Закрываем и удаляем все обработчики, которые держат файлы
for handler in logger.handlers[:]:
    handler.close()  # Закрывает сам файл внутри операционной системы
    logger.removeHandler(handler)  # Отвязывает его от логгера

Данные собирались в период 15.06.2026. На количество и качество данных влияли следующие факторы:
1. Сброс рангов и старт Нового Сезона. 28 апреля 2026 года официально завершился 1-й сезон («For Demacia») и начался 2-й сезон. В апреле 2006 был конец сезона и база игроков в лигах Challenger, Grandmaster, master была максимально заполнена. В начале июня 2026 - прошел всего месяц со старта нового сезона. Лиги Мастеров и Претендентов еще не успели заполниться людьми физически.
 
3. Квоты и ограничения на места в топе: Challenger и Grandmaster имеют жесткий лимит на количество мест (например, ровно 300 и 700 мест на регион). Но ранг Master лимита не имеет — туда попадают все, кто набрал нужное количество LP. В конце сезона в Мастерах находятся десятки тысяч игроков. В первый месяц сезона там находится лишь небольшая группа самых активных «киберспортсменов» и стримеров.
  
3. Ограничение пагинации. Поскольку количество игроков уменьшилось именно в самом сервере Riot, метод data.get("entries") выдает чистый, актуальный на сегодня список игроков, которые успели зайти в топ-лиги к маю 2026 года.

Ближе к июлю-августу 2026 года (к концу 2-го сезона) этот же скрипт снова начнет собирать по 45–50 тысяч игроков, так как Мастер-лига опять разрастется.

## Запрос информации о рейтинговых матчах за текущий месяц у 50 активных игроков по каждой лиге и региону

**Рейтинговым (ранговым)** считается матч, исход которого напрямую влияет на ранг игрока и его очки лиги (LP). В League of Legends есть три основных рейтинговых режима. 
Матч является рейтинговым, если его **queueId** равен одному из следующих чисел:
- 420 — Ranked Solo/Duo (Одиночная/Парная очередь на карте Ущелье призывателей). Самый популярный соревновательный режим.
- 440 — Ranked Flex (Гибкая очередь для команд на карте Ущелье призывателей).
- 1160 — Ranked 5v5 (Возвращенный командный режим для готовых групп из 5 человек).

Используем эндпоинт **GET /lol/match/v5/matches/by-puuid/{puuid}/ids**, который возвращает список матчей конкретного игрока (по puuid), в зависимости от переданных параметров.

Параметры:
- startTime: Epoch Timestamp начало периода
- endTime:  конец периода
- start - Начальный индекс (По умолчанию 0)
- count - Количество матчей (принимат значения от 0 до 100, по умолчанию 20).

**Для увеличения скорости сбора** ограничим количество игроков, для которых будем собирать информацию. 
На предыдущем шаге мы собрали данные 47 тысяч игроков по трем лигам и двум регионам. Теперь **отберем по 50 самых лучших и активных для каждой связки «регион + лига»** по двум ключевым показателям: очки LP (качество игры) и количество побед/матчей (активность). В Riot API за это отвечают поля leaguePoints (очки), wins (победы) и losses (поражения).

In [26]:
file_name = base_dir /  'all_players_data.csv'

In [27]:
# Т.к. скрипт запускается частями, то открываем полученный на предыдущем шаге датафрейм final_df с PUUID игроков, чтобы не собирать данные вторично
final_df = pd.read_csv(file_name, encoding='utf-8', encoding_errors='ignore')
display(f"Файл загружен успешно: {file_name} | Строк: {final_df.shape[0]}")

'Файл загружен успешно: d:\\datasets\\LOL\\all_players_data.csv | Строк: 21631'

In [28]:
# Создаем столбец общей активности (суммируем победы и поражения), если эти поля есть
if 'wins' in final_df.columns and 'losses' in final_df.columns:
    final_df['total_games'] = final_df['wins'] + final_df['losses']
else:
    # Если информации о поражениях нет, берем только победы как маркер активности
    final_df['total_games'] = final_df.get('wins', 0)

# Сортируем ВСЮ таблицу по приоритету:
# Сначала идут те, у кого больше всего очков LP (leaguePoints).
# При равных очках выше будут те, у кого сыграно больше матчей (total_games).
final_df_sorted = final_df.sort_values(
    by=['region', 'league_type', 'leaguePoints', 'total_games'],
    ascending=[True, True, False, False]
)

# Группируем по региону и лиге и забираем строго ТОП-50 из каждой группы
df_top = final_df_sorted.groupby(['region', 'league_type']).head(50).reset_index(drop=True)

# Проверяем результат
display("=== Статистика после фильтрации ===")
display(df_top.groupby(['region', 'league_type']).size())
display(f"\nОбщий размер новой таблицы: {df_top.shape[0]} игроков.")

# Перезаписываем переменную для скрипта матчей
limited_df = df_top

'=== Статистика после фильтрации ==='

region  league_type
euw1    challenger     50
        grandmaster    50
        master         50
na1     challenger     50
        grandmaster    50
        master         50
dtype: int64

'\nОбщий размер новой таблицы: 300 игроков.'

In [29]:
# Сохраним ТОП-50 игроков по каждой лдиге и региону в отдельный файл
file_name = base_dir /  'top_players.csv'
df_top.to_csv(file_name, mode='w', index=False, header=True, encoding="utf-8")

In [30]:
# Временной интервал - май 2006 
# Начало
start_of_month = datetime(2006, 5, 1, 0, 0, 0)
start_timestamp  = 1777592400  # 1 мая 2026 00:00:00
# Конец 
end_of_month = datetime(2006, 5, 31, 23, 59, 59)
end_timestamp = 1780270799    # 31 мая 2026 23:59:59

display(f"Старт: {start_of_month} -> {start_timestamp}")
display(f"Конец: {end_of_month} -> {end_timestamp}")

'Старт: 2006-05-01 00:00:00 -> 1777592400'

'Конец: 2006-05-31 23:59:59 -> 1780270799'

In [31]:
# Список PUUID  игроков
player_puuids = limited_df['puuid'] 
unique_matches = set()

Чтобы скрипт не потерял данные при возможном сбое сети или ошибке API, будем сохранять промежуточные итоги по каждому игроку в один общий датасет сразу после того, как закончим скачивание его истории. И далее будем сразу записывать этот обновленный датасет на диск (в CSV), реализуя полноценную систему чекпоинтов.

In [32]:
# Выходной файл
output_file_csv = base_dir /  'players_match_month.csv'

In [33]:
def count_lines(filename):
    """ Функция считаем количество строк в файле, не открывая его.
    Необходимо для минимизации расходуемой памяти
    """
    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        # Считаем перенос строки для каждой записи, не загружая текст в ОЗУ
        return sum(1 for _ in f)

In [34]:
# Проверяем существует ли уже файл на диске: если да, продолжим сбор с того места, где произошла остановка, если нет — создадим новый общий датасет.
if os.path.exists(output_file_csv):
    main_df = pd.read_csv(output_file_csv)
    # Собираем уже обработанные puuid, чтобы не запрашивать их повторно
    already_processed_players = set(main_df['puuid'].unique())
    output_rows = main_df.to_dict('records')
    print(f"Найдена существующая база данных. Игроков уже обработано: {len(already_processed_players)} players.")
else:
    already_processed_players = set() 
    with open(output_file_csv, "w", encoding="utf-8") as f:
        f.write("region,league_type,puuid,match_id,search_month,collected_at\n")  # Заголовки столбцов таблицы
        print(f"Файл {output_file_csv} не найден. Начинаем заново.")

Файл d:\datasets\LOL\players_match_month.csv не найден. Начинаем заново.


In [35]:
def collection_of_matches_for_the_month(puuid, player_index, league_type, region, my_headers):
    """
    Функция скачивает матчи для ОДНОГО игрока и сразу дописывает их в CSV на диск.
    Память полностью очищается после завершения работы функции.
    """
    display(f" Начинаем сбор для игрока: {player_index}: {puuid}")
        
    route = get_route(region) 
    
    # Запрашиваем сразу до 100 матчей за один запрос
    url = f"https://{route}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
       
    params = {
            "startTime": start_timestamp,
            "endTime": end_timestamp,
            "count": 100
            # в MATCH-V5 есть баг: совместное использование фильтров времени (startTime) и queue часто возвращает пустой список [].
            #"queue": 420,  Только Ranked Solo 
        }
        
    try:
        response = requests.get(url, headers=my_headers, params=params, timeout=5)
        
        if response.status_code == 429:
            logging.info("⚠️ Лимит запросов! Ждем 10 секунд...")
            time.sleep(10)
            return
            
        if not response.ok:
            logging.info(f"❌ Ошибка API для игрока {player_index}: {response.status_code}")
            return

        match_ids = response.json()  # Получили реальный список ID матчей от сервера
        
        if match_ids:
            temp_rows = []
            current_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            # Фиксируем месяц поиска (например, "May 2026") на основе переданного timestamp
            search_month_str = datetime.fromtimestamp(start_timestamp).strftime("%B %Y")

            for m_id in match_ids:
                temp_rows.append({
                    "region": region, 
                    "league_type": league_type,
                    "puuid": puuid,
                    "match_id": m_id,
                    "search_month": search_month_str,
                    "collected_at": current_time_str
                })
            
            # Создаем временный датафрейм только для ОДНОГО этого игрока
            player_df = pd.DataFrame(temp_rows)
            
            # Проверяем, существует ли файл. Если нет — пишем с заголовками, если да — просто дописываем строки
            file_exists = os.path.exists(output_file_csv)
            
            # Сохраняем на диск методом дозаписи (mode='a'), не нагружая оперативную память
            player_df.to_csv(output_file_csv, mode='a', index=False, header=not file_exists, encoding="utf-8")
            
            logging.info(f" Игрок {player_index}: Успешно сохранено {len(player_df)} матчей на диск.")
            
            # --- ПРИНУДИТЕЛЬНАЯ ОЧИСТКА ПАМЯТИ ---
            del temp_rows
            del player_df
            gc.collect()  # Сборщик мусора физически освобождает ОЗУ [gc]
               
        else:
            logging.info(f" Игрок {player_index}: Нет данных за период.")
            
    except requests.exceptions.Timeout:
        logging.info(f"❌ Превышено время ожидания для игрока {player_index}")
    except Exception as e:
        logging.info(f"❌ Сбой на игроке {player_index}: {e}")

In [36]:
log_file = base_dir /  'players_match_month.log'
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,  #Сбрасывает настройки Jupyter
)

In [37]:
display(f"Начинаем сбор матчей за период {start_of_month} -{end_of_month}")
logging.info(f"Начинаем сбор матчей за период {start_of_month} -{end_of_month}")

# По игрокам
for player_index, row in limited_df.iterrows():
    puuid = row['puuid']
    # Если игрок уже был успешно сохранен ранее — пропускаем его
    if puuid in already_processed_players:
        logging.info(f" Игрок {puuid[:15]}... уже обработан ранее. Пропуск.")
        continue
    league_type = row['league_type']
    region = row['region']
    collection_of_matches_for_the_month(puuid, player_index, league_type, region, HEADERS)
    
    # ПРИНУДИТЕЛЬНАЯ ОЧИСТКА ПАМЯТИ
    # Стираем временные связи текущей итерации и очищаем ОЗУ
    gc.collect() 
    
    # Соблюдаем лимит Riot API (1.2 - 1.5 секунды)
    time.sleep(1.4)
logging.info(f"Завершено. Лог файл - players_match_month.log")

'Начинаем сбор матчей за период 2006-05-01 00:00:00 -2006-05-31 23:59:59'

' Начинаем сбор для игрока: 0: 7qDgswqpOmHpVyAQO-085TwvVSp5kuojnI7ehOzZiBQX4JksjCF3bVeM-9UyMQXuYNJbX-Gn3gKeog'

' Начинаем сбор для игрока: 1: D8z6JW2Ea_FfL6zRR98wcFmaMpK1KZRbYBGS3pIySYqNC06W4ai6nN3U_O2iv9ML6IS01wsYsIpI7w'

' Начинаем сбор для игрока: 2: Sb7usV3fHduwHxZoXjwHdaVxF1T2LhaBHejvJdqOvRhZv_pL3fWVrgjwh9DDpM-oS-LBVGFhrn4y0g'

' Начинаем сбор для игрока: 3: ZIemTzkIPML6oScobsLKDZZCmGXzGA9yHbJAymoaqRNFopIfvWFpdcrc38PNm1p0XFOI67dvaqnzRw'

...

' Начинаем сбор для игрока: 295: gwn5_V2kf2nZiG0PzUF2j7XHwbdXh8atNTAbUE_Sq9Y5OPLi4RBuBF4vXIuD0fgD8l4OMTKqIu_yiA'

' Начинаем сбор для игрока: 296: -zB6utkbHZco62V1vFz_TrYuBuSfF4vQOA81oQF-M2aQ53yO6jzuDmBRn6ZTd1NZrLx-fOGdaYpHxg'

' Начинаем сбор для игрока: 297: 7SKhPc0qqviaiR7eGTGyXWFa8FtwTla8vqcEiv9St-tRQrs5VhdToU6DcZcd-gd00SPToZoUuf2Phg'

' Начинаем сбор для игрока: 298: 4c7aS4_QyQTFv9gPe2btNhpkHCA9XQ9vBPCjLWXWDgH04Ts3Q3NVOQ5N7_lEsIaY0Hblg7I9aO4qZw'

' Начинаем сбор для игрока: 299: THUnzdWHKm8Zu_7LjRroBGnCPF6pFJXUPN2E_6QvFwgdNZPk4vlQVmpAEAIvJa3np_BHblo0iBT_Rw'

In [38]:
logger = logging.getLogger()

# Закрываем и удаляем все обработчики, которые держат файлы
for handler in logger.handlers[:]:
    handler.close()  # Закрывает сам файл внутри операционной системы
    logger.removeHandler(handler)  # Отвязывает его от логгера

In [39]:
total_lines = count_lines(output_file_csv)
display(f"Количество собранных матчей по игрокам: {total_lines - 1}")  # Вычитаем строку заголовка

'Количество собранных матчей по игрокам: 28268'

## Выборка уникальных ID матчей за текущий месяц

На предыдущем шаге мы получили файл с ID матчей (до 100 штук) за месяц для самых активных игроков трех лиг и двух регионов. Но, т.к. игроки могут играть друг против друга в одних и тех же матчах и  ID матчей при этом совпадают, оставим только уникальные ID матчей, в дальнейшем получим по ним более детальную информацию.

In [61]:
# файл с матчами
output_file_csv = base_dir /  'players_match_month.csv'

In [62]:
# Открываем файл с ID матчами
try:
    df = pd.read_csv(output_file_csv)
    display(f"Файл загружен успешно: {output_file_csv} | Строк: {df.shape[0]}")
except Exception as e:
    print(f"Ошибка при загрузке: {output_file_csv}: {e}")

'Файл загружен успешно: d:\\datasets\\LOL\\players_match_month.csv | Строк: 28268'

In [63]:
df_unique_matches = pd.DataFrame(df['match_id'].unique(), columns=['match_id'])

In [64]:
print(f'Всего уникальных матчей за месяц у активных игроков: {df_unique_matches.shape[0]}')

Всего уникальных матчей за месяц у активных игроков: 23329


In [65]:
# Сохраним в отдельный файл
output_file_csv = base_dir / "id_unique_matches.csv" 
df_unique_matches.to_csv(output_file_csv, index=False)

## Запрос детальной информации: матчи и игроки

Для этой цели используем эндпоинт **GET /lol/match/v5/matches/{matchId}**, который возвращает информацию о матче по его ID.

**Для матчей** выборочная структура объектов ответа:
- info.gameDuration (long) - длительность матча в секундах
- info.gameCreation (long) - Unix timestamp время, когда игра создана на сервере
- info.gameMode(string) -  режим игры
- info.gameVersion (string) - версия/патч игры

В игре каждый матч — это битва двух команд, где каждый из 10 участников (participants) перед стартом матча выбирает себе одного уникального игрового персонажа, т.н. "чемпиона". 

**Для игроков** выбрираем объекты:
- info.participants.puuid (string) - идентификатор игрока
- info.participants.summonerName (string) - имя - УСТАРЕЛО!!!
- info.participants.kills (int) - убийства
- info.participants.deaths (int) - смерти
- info.participants.assists (int)  - помощь
- info.participants.win (boolean) - 1 - Победа, 0 - Поражение
- info.participants.teamId (int) - 100 (Синяя команда)/200 (Красная команда) 
- info.participants.championId (int)-  уникальный числовой идентификатор используемого чемпиона

Вместо **summonerName** используются **riotIdGameName** (основной игровой ник, виден всем игрокам, может повторяться) и **riotIdTagline** (уникальная метка, позволяет системе отличать игроков с одинаковыми никами). 

Поле **summonerLevel** полностью удалено из истории матчей, уровень постоянно меняется, а матч фиксирует историю. В MATCH-V5 его больше нет.

**Создадим 2 таблицы:**
- Таблица матчей - содержит общую информацию (ID игры, длительность, версия патча).
- Таблица участников (participants): где под одним match_id создается 10 строк (по одной на каждого игрока) с их личной статистикой.

Ниже - скрипт, который берет сохраненные matchId из файла, запрашивает подробные данные по каждому матчу через эндпоинт /lol/match/v5/matches/{matchId} и раскладывает их на три отдельные таблицы: Матчи, Игроки (Участники) и Чемпионы.

Чтобы полностью исключить ошибку Out of Memory, код будет использовать потоковую обработку: парсить по одному матчу за раз, мгновенно записывать новые строки в три CSV-файла в режиме дозаписи (mode='a') и принудительно очищать оперативную память через gc.collect().

In [66]:
input_match_ids_csv = base_dir / "id_unique_matches.csv"   # Файл с ID матчей
output_matches_csv = base_dir / "matches_data.csv"         # Таблица 1: Матчи
output_players_csv = base_dir / "players_data.csv"         # Таблица 2: Участники

In [67]:
# Загрузка исходных ID матчей
if not os.path.exists(input_match_ids_csv):
    display(f"Ошибка! Файл {input_match_ids_csv} не найден.")
    exit()

df_source = pd.read_csv(input_match_ids_csv)
all_match_ids =  set(df_source['match_id'].dropna())
display(f"Загружено уникальных матчей для обработки: {len(all_match_ids)}")

'Загружено уникальных матчей для обработки: 23329'

In [68]:
# 1. Загрузка исходных ID
if not os.path.isfile(input_match_ids_csv):
    raise FileNotFoundError(f"Исходный файл не найден: {input_match_ids_csv}")

try:
    df_source = pd.read_csv(input_match_ids_csv, usecols=['match_id'])
    unique_match_ids = set(df_source['match_id'].dropna().unique())
    print(f"Всего исходных матчей: {len(unique_match_ids)}")
except KeyError:
    raise KeyError("В исходном файле отсутствует колонка 'match_id'")

# 2. Фильтрация уже обработанных ID
processed_ids = set()
if os.path.isfile(output_matches_csv) and os.path.getsize(output_matches_csv) > 0:
    try:
        df_processed = pd.read_csv(output_matches_csv, usecols=['match_id'])
        processed_ids = set(df_processed['match_id'].dropna().unique())
        print(f"Уже обработано ранее: {len(processed_ids)}")
    except KeyError:
        print("В файле результатов нет 'match_id'. Обработка начнется с нуля.")

# 3. Вычисление остатка
unique_match_ids = list(all_match_ids - processed_ids)
print(f"Осталось обработать пропущенных матчей: {len(unique_match_ids)}")

if not unique_match_ids:
    print("Все матчи уже успешно обработаны!")

Всего исходных матчей: 23329
Уже обработано ранее: 19585
Осталось обработать пропущенных матчей: 12386


In [69]:
def save_batch_to_csv(batch_matches, batch_players):
    """
    Принимает списки словарей для матчей и игроков, 
    переводит в DataFrame и дописывает в CSV.
    """
    if not batch_matches or not batch_players:
        return

    # 1. Сохраняем батч матчей
    df_matches = pd.DataFrame(batch_matches)
    matches_exist = os.path.isfile(output_matches_csv)
    df_matches.to_csv(output_matches_csv, mode='a', index=False, header=not matches_exist, encoding='utf-8')

    # 2. Сохраняем батч игроков (здесь будет ~1000 строк)
    df_players = pd.DataFrame(batch_players)
    players_exist = os.path.isfile(output_players_csv)
    df_players.to_csv(output_players_csv, mode='a', index=False, header=not players_exist, encoding='utf-8')

    print(f"💾 Батч успешно записан на диск ({len(batch_matches)} матчей, {len(batch_players)} игроков).")


In [70]:
def save_match_batch_to_csv(batch_list, file_path="matches_data.csv"):
    """
    Принимает список из 100 JSON-объектов матчей, 
    выпрямляет их и дописывает в CSV-файл.
    """
    if not batch_list:
        return
        
    all_participants = []
    
    # 1. Разворачиваем вложенный JSON матчей для CSV
    for match in batch_list:
        match_id = match['metadata']['matchId']
        game_version = match['info']['gameVersion']
        
        # Вытаскиваем данные всех 10 игроков из этого матча
        for participant in match['info']['participants']:
            # Создаем плоский словарь для строки CSV
            player_row = {
                'match_id': match_id,
                'game_version': game_version,
                'puuid': participant['puuid'],
                'champion_name': participant['championName'],
                'win': participant['win'],  # Наше булево поле!
                'kills': participant['kills'],
                'deaths': participant['deaths'],
                'assists': participant['assists']
                # Добавьте сюда любые другие нужные вам поля
            }
            all_participants.append(player_row)
            
    # 2. Переводим накопленный батч в DataFrame
    df_batch = pd.DataFrame(all_participants)
    
    # 3. Записываем в файл в режиме добавления (mode='a')
    # Если файл еще не создан, пишем его с заголовками (header=True)
    file_exists = os.path.isfile(file_path)
    
    df_batch.to_csv(file_path, mode='a', index=False, header=not file_exists, encoding='utf-8')
    print(f"💾 Батч успешно дозаписан в CSV. Файл: {file_path}")


In [71]:
def match_data_collection(match_id, match_index):
    """
    Запрашивает один матч, парсит его на 2 структуры и возвращает их без записи на диск.
    """
    region_prefix = match_id.split('_')[0].lower()
    route = get_route(region_prefix) 
    url = f"https://{route}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=5)
        
        if response.status_code == 429:
            print("⚠️ Лимит запросов! Засыпаем на 12 секунд...")
            time.sleep(12)
            return None
            
        if not response.ok:
            print(f"❌ Ошибка API для матча {match_id}: Код {response.status_code}")
            return None

        match_data = response.json()
        info = match_data.get("info", {})
        
        if not info:
            return None

        # --- 1. ПАРСИНГ ДАННЫХ МАТЧА ---
        match_row = {
            "match_id": match_id,
            "game_name":  info.get("gameName"), 
            "game_creation": info.get("gameCreation"),
            "game_duration": info.get("gameDuration"),
            "game_version": info.get("gameVersion"),
            "game_mode": info.get("gameMode"),
            "endOfGameResult": info.get("endOfGameResult"),
            "gameStartTimestamp": info.get("gameStartTimestamp"),
            "gameEndTimestamp": info.get("gameEndTimestamp"),
            "gameType": info.get("gameType"),
            "platform_id" : info.get("platformId"),
            "queue_id": info.get("queueId")
        }

        # --- 2. ПАРСИНГ ИГРОКОВ (10 участников на матч) ---
        match_players = []
        participants = info.get("participants", [])
        for p in participants:
            player_row = {
                "match_id": match_id,
                "puuid": p.get("puuid"),
                #"summoner_name": p.get("summonerName"),   # Устарело
                #"summoner_level": p.get("summonerLevel"), # Устарело
                "riotIdGameName" : p.get("riotIdGameName"),
                "riotIdTagline" :  p.get("riotIdTagline"),
                "kills": p.get("kills"),
                "deaths": p.get("deaths"),
                "assists": p.get("assists"),
                "win": p.get("win"),
                "teamPosition": p.get("teamPosition"),
                "goldEarned": p.get("goldEarned"),
                "goldSpent": p.get("goldSpent"),
                "team_id": p.get("teamId"),
                "champion_id": p.get("championId"),
                "championName":  p.get("championName"),
                "timePlayed": p.get("timePlayed")
            }
            match_players.append(player_row)

        print(f"👁️ Матч {match_index} - {match_id} обработан локально.")
        
        # Возвращаем кортеж: строка матча и список строк его игроков
        return match_row, match_players

    except Exception as e:
        print(f"❌ Критическая ошибка при обработке матча {match_id}: {e}")
        return None


In [72]:
# --- ГЛАВНЫЙ ПОТОКОВЫЙ ИТЕРАТОР (батчи по 100 штук)
def main_pipeline():
    print("🚀 Запуск потокового парсинга матчей пачками по 100 штук...")
    
    batch_size = 100
    accumulated_matches = []
    accumulated_players = []
    
    for index, match_id in enumerate(unique_match_ids, start=1):
        match_id = str(match_id).strip()
        if not match_id:
            continue
               
        # Вызываем парсер
        result = match_data_collection(match_id, index)
            
        if result:
            match_row, match_players = result
            accumulated_matches.append(match_row)
            accumulated_players.extend(match_players) # Разворачиваем список из 10 игроков
            
        # Если набрали 100 матчей ИЛИ это конец списка уникальных ID
        if len(accumulated_matches) == batch_size or index == len(unique_match_ids):
            print(f"📦 Накоплен батч! Запись {len(accumulated_matches)} матчей...")
            
            # Записываем пачку на диск за один раз
            save_batch_to_csv(accumulated_matches, accumulated_players)
            
            # Очищаем списки и принудительно освобождаем RAM
            accumulated_matches.clear()
            accumulated_players.clear()
            gc.collect() 
            
        # Rate Limit Riot API (1.2 сек — надежный тайминг для 1 запроса)
        time.sleep(1.2)
        
    print("🏁 Весь процесс парсинга успешно завершен!")


In [73]:
main_pipeline()
display("Сбор данных закончен")

🚀 Запуск потокового парсинга матчей пачками по 100 штук...
👁️ Матч 1 - NA1_5562266569 обработан локально.
👁️ Матч 2 - NA1_5568787128 обработан локально.
👁️ Матч 3 - NA1_5570800772 обработан локально.
👁️ Матч 4 - NA1_5568372832 обработан локально.
👁️ Матч 5 - EUW1_7862698204 обработан локально.
👁️ Матч 6 - NA1_5567754321 обработан локально.
👁️ Матч 7 - NA1_5570698001 обработан локально.
👁️ Матч 8 - EUW1_7871699959 обработан локально.
👁️ Матч 9 - NA1_5570886685 обработан локально.
👁️ Матч 10 - EUW1_7869785584 обработан локально.
👁️ Матч 11 - NA1_5565685202 обработан локально.
👁️ Матч 12 - EUW1_7867066341 обработан локально.
👁️ Матч 13 - NA1_5563167505 обработан локально.
👁️ Матч 14 - NA1_5568647804 обработан локально.
👁️ Матч 15 - NA1_5572341321 обработан локально.
👁️ Матч 16 - NA1_5564207578 обработан локально.
👁️ Матч 17 - NA1_5570654226 обработан локально.
👁️ Матч 18 - EUW1_7861001289 обработан локально.
👁️ Матч 19 - EUW1_7859055947 обработан локально.
👁️ Матч 20 - EUW1_7852160968 обр

'Сбор данных закончен'

## Вывод

В период 14.06.2026-16.06.2026 с помощью официального API Riot Games были собраны следующие данные за май 2026:

- **all_players_data.csv** - 21631 записей об игроках в лигах Challenger, Grandmaster, Master для регионов euw1 и na1 с актуальными данными по игроку на 15.06.2026
  
- из all_players_data.csv выбрано по 50 самых активных игроков по каждой лиге и региону - получен файл **top_players.csv** (300 записей)

- для каждого из 300 игроков из top_players.csv получен список до 100 матчей, в которых игрок участвовал в мае 2026 года - **players_match_month.csv** (28269 записей)

- для каждого матча из players_match_month.csv с помощью API получена детальная информация о матчах - **matches_data.csv** (31965 записей)

- для каждого матча из matches_data.csv получены данные по игрокам, участвующих в матчах -  **players_data.csv** (343524 записи)